# Feature selection

Autores: Pablo Hernández Cámara (pablo.hernandez-camara@uv.es) y Pedro Ramón Ventura Gómez (pventura@march-am.com)



En este notebook vamos a ver como se pueden utilizar los métodos de feature selection para reducir el número de características.

Recordad que la selección de características tiene como objetivo reducir el número de variables de nuestros datos seleccionando solo algunas de las variables existentes.

## Filter methods

En particular, los métodos filter se basan en hacer un ranking de todas las variables de entrada según cierto criterio elegido para quedarse solo con las que obtengan una mejor puntuación en dicho ranking.

Estos métodos tienen la ventaja de ser muy rápidos y fáciles de implementar y son independientes del modelo que se quiera utilizar posteriormente.

In [ ]:
from sklearn import datasets
import matplotlib.pyplot as plt
%matplotlib inline
from scipy import stats
import pandas as pd
import numpy as np

Vamos a utilizar el conjunto de datos de Iris, uno de los más famosos en machine learning para hacer problemas pequeños y probar métodos/modelos.

In [ ]:
iris = datasets.load_iris()

**Ejercicio: ¿Que contiene la variable iris que acabas de cargar?**

**Ejercicio: Guarda en una variable ``X`` los datos y en una variable ``y`` la variable a predecir. Comprueba las dimensiones**

Antes de comenzar con los métodos filter, vamos a hacer un pequeño analisis de los datos. Esta etapa de preprocesado/analsis de nuestros datos, es una de las más importantes en machine learning, ya que nos permite obtener conocimiento de nuestros datos para tomar decisiones correctas posteriormente.

Hay multitud de formas de hacerlo, una de las más sencillas es convirtir los datos a un dataframe de pandas, lo que nos permite facilmente obtener estadisticos y descriptores por ejemplo.

In [ ]:
iris_data = pd.DataFrame(X, columns=iris.feature_names)
iris_data['target'] = y
iris_data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [ ]:
print(f"Missing Values: {iris_data.isnull().sum().sum()}")

iris_data.describe()

Missing Values: 0


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


In [ ]:
iris_data['target'].value_counts()

,count
target,
0,50
1,50
2,50


Otro paso que aquí no hemos hecho pero que es importante hacer es la normalización/estandarización de los datos. En general, los algoritmos funcionan mejor (y son mucho más interpretables) cuando las variables tienen todas la misma escala.

Hay que tener en cuenta que distintos tipos de normalización (MinMax Scaler, Absolute Scaler, Standarizacion...) harán distintas transformaciones a nuestros datos, y según como sean, unas pueden ser mejores que otras.

Como hemos comentado existen distintas métricas que podemos usar para hacer el ranking de variables y seleccionar las que queremos mantener.

Una primera opción es calcular distintos estadísticos individuales de cada variable para construir el ranking. En este caso, la idea es mantener aquellas variables más dispersas, ya que al estar menos concentradas tendrán más información. Ejemplos posibles son la desviación estandar, el rango intercuartil o el ratio de dispersión, que se define como el ratio entre la media aritmetica y la media geométrica.

**Ejercicio: Calcula para cada variable: 1) desviación estandar, 2) rango intercualtilico y 3) ratio de dispersión. Razona en cada caso con que variables nos quedariamos si quisieramos mantener solo dos de las 4.**

Estos ejemplos usan cada variable de forma independiente y aunque son muy fáciles de calcular, no recogen información de como interactuan entre ellas.

Para ello, hemos de utilizar métodos para comparar como se relacionan las variables entre sí. En este caso, la idea será buscar si hay algunas variables muy relacionadas entre sí para mantener únicamente una de ellas. Para hacer esto, podemos utilizar distintas métricas, por ejemplo, calcular la correlación entre cada par de variables.

**Ejercicio: Calcula y representa la matriz de correlaciones entre las 4 variables de entrada. Puedes usar ``np.corrcoef``. ¿Eliminarias alguna variable?**

Fijaros que hasta aquí no hemos utilizado para nada la variable a predecir ``y``, por eso estos métodos son no supervisados. Sin embargo, se pueden usar también cuando disponemos de una variable a predecir, de forma supervisada. En este caso, podemos calcular no solo la relación entre las variables de entrada, si no también entre cada variable de entrada y la variable a predecir.

**Ejercicio: Calcula en este caso las correlaciones entre cada variable y la variable de salida. Utiliza esta información para razonar cual de las dos variables anteriores sería mejor mantener.**

De esta forma, si tenemos dos variables de entrada muy relacionadas, podemos ver cual nos convendrá más mantener.

Pero ojo, que tipo de correlación estamos calculando? Busca que es lo que hace exactamente ``np.corrcoef``. ¿Que correlación es? ¿Que detecta esta medida?

Para capturar relaciones no lineales entre las variables, hemos de utilizar otra medida. Por ejemplo, correlación de Spearman, que captura relaciones monotonas o, mejor aun, usando teoría de la información podemos calcular la información mútua entre variables.

Vamos a calcular la MI. La MI entre dos variables **discretas** $X,Y$ viene dada por
$$ \textrm{MI}(X,Y) = \sum_{y\in \mathcal{Y}}\sum_{x\in \mathcal{X}} p(x,y)\log\left(\frac{p(x,y)}{p(x)p(y)}\right) $$
Más info: https://en.wikipedia.org/wiki/Mutual_information


En primer lugar estimamos la probabilidad para cada clase de la variable de salida. También podriamos calcular la entropia de cada variable, viene dada por $H(y)=-\sum p(y)\log p(y) $ y usarla como descriptor para decidir que variables mantener.

In [ ]:
# Entropia de las variables //discretas// de salida: H(y) = -sum P(c)log(P(c))
classes = np.unique(y)
print(classes)

# Probablidad de cada classe
p_y = np.zeros(len(classes))
for nc, c in enumerate(classes):
    p_y[c] = np.sum(y == c)
# Convertimos en probablididades normalizando por el total
p_y /= np.sum(p_y)
print(p_y)
print('\n')

# Calculamos la entropía
h_y = -np.sum(p_y * np.log(p_y))
print('p(y)=', p_y, '\ny entropy=', h_y)

[0 1 2]
[0.33333333 0.33333333 0.33333333]


p(y)= [0.33333333 0.33333333 0.33333333] 
y entropy= 1.0986122886681096


Para poder trabajar con las variables de la entrada tendremos que cuantificarlas dado que son continuas (también se puede hacer con continuas, pero haciendolo a mano es bastante más complicado). Vamos a utilizar `KBinsDiscretizer` unas nuevas variables `Xd` discretizando la variable `X`, utilizando 9 bins.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

kbd = KBinsDiscretizer(n_bins=9, encode='ordinal')
Xd = kbd.fit_transform(X)

El siguiente código estima p(x) y p(x,y) y así podemos calcular la MI entre cada variable de entrada y la salida.

In [ ]:
# Vamos a calcular la MI para cada variable en X
nv = Xd.shape[1]
mi = np.zeros(nv)

# Valores distintos en Xd
xvalues = np.unique(Xd)

for v in range(nv):
    # Calculamos px
    px = np.zeros(len(xvalues))
    # --------------
    for i, value in enumerate(xvalues):
        px[i] = np.sum([Xd[:,v] == value])
    px /= np.sum(px)
    # --------------

    # Calculamos pxy
    pxy = np.zeros((len(xvalues), len(classes)))
    # --------------
    for i, value in enumerate(xvalues):
        for j, cls in enumerate(classes):
            pxy[i,j] = np.sum((Xd[:,v] == value) & (y == cls))
    pxy /= np.sum(pxy)
    # --------------

    # Calculamos MI: sum_x(sum_y( p(x,y) * log(p(x,y) / (p(x)*p(y)) ))
    mi[v] = 0.0
    for i in range(pxy.shape[0]):
        for j in range(pxy.shape[1]):
            # Evitar divisiones por cero y logaritmos = -inf
            if pxy[i,j] > 0 and px[i] > 0 and p_y[j] > 0:
                # --------------
                mi[v] += pxy[i,j] * np.log(pxy[i,j] / px[i] / p_y[j])
                # --------------

print(mi)

[0.5032742  0.31671819 0.98041703 0.97135001]


**Ejercicio: Usando esta técnica hemos obtenido resultados distintos. ¿Que quiere decir un valor alto? ¿Cual de las dos variables eliminariamos ahora?**

Esto que hemos hecho "a mano", hay funciones que nos lo hacen directamente. Vamos a verlo:

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X, y, discrete_features='auto')  # random_state asegura repetitividad
print(mi)

[0.51549257 0.29047958 0.99435297 0.99406783]


**Ejercicio: Calcula ahora la matriz de información mutua entre las variables de entrada. Fijate que igual que existe ``mutual_info_classif``, también existe ``mutual_info_regression``.**

De hecho, ``sklearn`` tiene también una función que nos realiza la selección de las mejores variables de forma automática, donde solo hemos de especificar la métrica que queramos que se use para realizar el ranking.

**Ejercicio: Busca y aplica la función ``SelectKBest`` para obtener el conjunto de datos con solo 2 variables donde se hayan mantenido aquellas con menor información mutua con la variable a predecir.**

## Wrapper methods

Los métodos wrapper se basan en seleccionar las variables que mejor funcionarán para un determinado modelo. Por ello, es necesario tener ya claro que modelo se quiere usar, en este caso que tenemos etiquetas, vamos a usar uno de clasificación, pero también se podrian usar con modelos no supervisados que veremos mañana.

Estos métodos tienen la ventaja de ser específicos para el modelo que se quiere usar y por tanto dar mejores resultados que los métodos filter. Sin embargo, también son dependientes del módelo usado y las variables elegidas para un cierto modelo no tienen por que ser las mismas para otro. Además, en general son mucho más costosos computacionalmente, ya que implican entrenar los modelos varias veces.

In [ ]:
from sklearn.datasets import make_regression

# Para que el ejemplo tenga sentido tenemos que poner n_informative < n_features
X, y = make_regression(n_samples=300, n_features=4, n_informative=3, n_targets=1, noise=0.01)

print(X.shape, y.shape)

(300, 4) (300,)


En este caso vamos a elegir como modelo una regresión lineal, que usaremos a partir de ``sklearn``. Pero primero vamos a hacer un paso imprescindible, separar los datos.

In [ ]:
from sklearn.model_selection import train_test_split

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.33) #stratify
print(Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape)

(201, 4) (99, 4) (201,) (99,)


In [ ]:
# Nos aseguramos de que la y tenga dimensión Nx1
ytrain = ytrain.reshape(-1,1)
ytest = ytest.reshape(-1,1)
print(ytrain.shape, ytest.shape)

(201, 1) (99, 1)


Vamos a ver que resultado obtenemos primero con todas las variables.

**Ejercicio: Entrena una regresion lineal en los datos.**

**Ejercicio: Evalua el Mean Square Error del modelo en test.**

Vamos ahora con el método wrapper propiamente dicho. En primer lugar vamos probando con cada variable una por una ...


**Ejercicio: Prueba una por una las variables de entrada utilizando  ``LinearRegression``. Comprueba los resultados con las funciones ``fit``, ``predict`` y  el MSE del modelo en test. ¿Cuál es la variable que proporciona el mejor modelo?**

Imaginemos que ahora queremos quedarnos con solo dos variables, ¿cuantas combinaciones posibles hay? Hay que entrenar un modelo para cada combinación! Por suerte, ``combinations`` de la librería ``itertools`` nos da todas las combinaciones.

**Ejercicio: Entrena un modelo en cada combinación de par de variables. ¿Cual es la mejor combinación?**

In [ ]:
from itertools import combinations


**Ejercicio: Y en grupos de 3 variables, ¿que variable descartamos?**

## Embedded methods

Estos métodos se basan en la propia explicabilidad que proporcionan algunos tipos de modelos. Vamos a verlo en un caso muy sencillo de una regresión lineal, aunque este tipo de métodos se puede aplicar a otros modelos, como los basados en árboles que también proporcionan información de la importancia de las variables.

In [ ]:
X, y = make_regression(n_samples=300, n_features=4)
y = y.reshape(-1, 1)
print(X.shape, y.shape)

(300, 4) (300, 1)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(210, 4) (90, 4) (210, 1) (90, 1)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
y_pred = lr.predict(X_test_s)

print(f'MSE = {np.mean((y_test - y_pred)**2)}')

MSE = 1.098988422426899e-26


Como deciamos, la regresión lineal no es más que una multiplicación de cada variable por un coeficiente. Estos coeficientes son la importancia que le está dando el modelo a cada variable y podemos utilizar esta información para eliminar las variables menos importantes.

**Ejercicio: Explora los coeficientes de la regresión. Si tuvieras que eliminar una variable para quedarte solo con 3 variables, ¿cual elegirias? ¿En que te has basado?**

**Ejercicio: Entrena ahora un arbol de regresión y obten la importancia de las variables. ¿Con cuales te quedarias en este caso?**